<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Several seasons and long series

**[Several seasons and long series](https://learning.nextia-ai.com/courses/forecasting/m04/several-seasons-and-long-series/)** · Forecasting and Time Series: Predict What Comes Next · Module 4, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** see the daily and the weekly season of the hourly help-desk tickets, split them into parts with MSTL, backtest MSTL and Theta on the course's ticket backtest, check how often the real tickets fall inside each range, and add a known event as an extra input.

| | |
|---|---|
| **Time** | About 35 minutes, of which about 20 seconds of waiting for the backtest |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | ETS, ARIMA and the backtest table, from [Exponential smoothing (ETS, Holt-Winters)](https://learning.nextia-ai.com/courses/forecasting/m04/exponential-smoothing/) and [ARIMA and SARIMA](https://learning.nextia-ai.com/courses/forecasting/m04/arima-and-sarima/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m04/several-seasons-and-long-series/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M04-L03-several-seasons-and-long-series/several-seasons-and-long-series-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Two seasons in one series

The help desk gets tickets by email, chat and phone. The next cell adds the three channels up for every hour, and shows the mean tickets per hour of the day and per day of the week.

> **Predict.** Which is larger: the difference between night and late morning, or the difference between Monday and Sunday?

In [ ]:
hourly = pd.read_csv("data/tickets_hourly.csv.gz", parse_dates=["timestamp"]).groupby("timestamp")["tickets"].sum()
tickets = pd.DataFrame({"unique_id": "tickets", "ds": hourly.index, "y": hourly.to_numpy(dtype=float)})
by_hour = tickets.groupby(tickets["ds"].dt.hour)["y"].mean()
by_day = tickets.groupby(tickets["ds"].dt.day_name())["y"].sum() / tickets["ds"].dt.normalize().nunique() * 7
print("mean tickets at 03:00:", round(by_hour[3], 1), "| at 11:00:", round(by_hour[11], 1))
print("mean tickets on a Monday:", round(by_day["Monday"]), "| on a Sunday:", round(by_day["Sunday"]))

> **Check.** You should see about 2.5 tickets at 03:00 and 32.0 at 11:00, and about 390 tickets on a Monday against 248 on a Sunday.

Both patterns repeat. The day repeats every 24 hours. The week repeats every 168 hours (7 × 24). One ETS model takes only one season length, so it must miss one of them.

## 2. MSTL: split the series into parts

**MSTL** (Multiple Seasonal-Trend decomposition using LOESS) splits the series into a trend, one part per season and a remainder. The next cell fits it to the last 12 weeks before Sunday 15 March 2026, 23:00, and shows how much of the up and down each part explains (its share of the variance).

In [ ]:
from statsforecast.models import MSTL, AutoETS

origin = pd.Timestamp("2026-03-15 23:00")
last12 = tickets[tickets["ds"] <= origin].tail(2016)  # 12 weeks x 168 hours
mstl = MSTL(season_length=[24, 168], trend_forecaster=AutoETS(model="ZZN"))
mstl.fit(last12["y"].to_numpy())
parts = mstl.model_
share = {c: round(float(100 * parts[c].var() / parts["data"].var()), 1) for c in ["trend", "seasonal24", "seasonal168", "remainder"]}
print(share)

> **Check.** You should see `{'trend': 0.4, 'seasonal24': 76.8, 'seasonal168': 8.7, 'remainder': 11.3}`.

The daily season is the largest part. The weekly season is smaller, but it is what tells a Monday from a Sunday. The parts do not add up to exactly 100%, because they are not fully independent. The chart shows the last week of each part.

In [ ]:
import matplotlib.pyplot as plt

week = parts.tail(168).reset_index(drop=True)
fig, axes = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
for ax, c in zip(axes, ["data", "seasonal24", "seasonal168", "trend"]):
    ax.plot(week.index, week[c])
    ax.set_ylabel(c)
axes[-1].set_xlabel("hour of the last week (0 = Monday 00:00)")
plt.show()

## 3. MSTL and Theta in the course backtest

To forecast, MSTL forecasts the trend (here with ETS without a season) and repeats the two seasons. **Theta** is another simple method: a line through the trend, combined with exponential smoothing, on the series without its season. It takes one season (here 168 hours).

The next cell runs the course's ticket backtest: 13 origins at 23:00, 168 hours (one week) after each, with the last 12 weeks of history. It keeps each method's 80% range (the columns `q10` and `q90`).

> **Predict.** Which has the lower WAPE: MSTL or Theta? Then run the cell.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoTheta

def with_range(model, keep=2016):
    def forecaster(train, h, future):
        sf = StatsForecast(models=[model], freq="h")
        out = sf.forecast(df=train[["unique_id", "ds", "y"]].tail(keep), h=h, level=[80])
        name = str(model)
        return pd.DataFrame({"unique_id": out["unique_id"], "ds": out["ds"], "yhat": out[name].clip(lower=0),
                             "q10": out[f"{name}-lo-80"].clip(lower=0), "q90": out[f"{name}-hi-80"].clip(lower=0)})
    return forecaster

TICKET_ORIGINS = [o + pd.Timedelta(hours=23) for o in ORIGINS]
fc_same = bt.backtest(tickets, bt.seasonal_naive(168), h=168, origins=TICKET_ORIGINS)
fc_mstl = bt.backtest(tickets, with_range(MSTL(season_length=[24, 168], trend_forecaster=AutoETS(model="ZZN"))),
                      h=168, origins=TICKET_ORIGINS)
fc_theta = bt.backtest(tickets, with_range(AutoTheta(season_length=168)), h=168, origins=TICKET_ORIGINS)
rows = [bt.score(f, tickets, name, season=168, cost=bt.roster_cost())
        for name, f in [("same hour last week", fc_same), ("MSTL", fc_mstl), ("Theta", fc_theta)]]
table = bt.show(rows)

> **Check.** You should see Theta first with MASE 0.883 and WAPE % 27.1, then MSTL with 0.919 and 28.3, then the same hour last week with 1.036 and 31.9. The cost column is the roster cost per week, in dollars. The pinball column scores the range; Module 7 explains it.

Theta has the lowest error, and its roster costs the least when Grace staffs exactly to the forecast. But a forecast is also a range. Is the range right?

## 4. Your turn: how often is the real value inside the range?

An 80% range should hold the real value in about 80% of the hours. More is not better: a range that always holds the real value is too wide to plan with.

> **Your turn.** For `fc_theta`, compute the share of hours where `q10 <= y <= q90`. Round it to 3 decimals and put it in `theta_coverage`. Then run the check cell.

In [ ]:
theta_coverage = ...  # the share of hours inside Theta's 80% range

In [ ]:
expect('theta_coverage', theta_coverage, 0.996)

Theta's "80%" range holds the real value in 99.6% of the hours: it is far too wide. MSTL's holds it in 85.1%: a little wide, close to right.

## 5. The roster at the 72% point of the range

Grace does not staff to the forecast. An hour one agent short costs about 72 dollars, and an extra agent hour 28 dollars (the course's assumptions). So she staffs to a point that is above the forecast 72% of the time: 72 ÷ (72 + 28). Module 7 explains why. The next cell computes that point from each range, and the roster cost.

In [ ]:
from scipy.stats import norm

def roster_at(fc, share=0.72):
    sd = (fc["q90"] - fc["q10"]) / (2 * norm.ppf(0.9))  # the spread of a normal range with these 10% and 90% points
    plan = (fc["yhat"] + norm.ppf(share) * sd).clip(lower=0)
    return round(bt.roster_cost()(fc.assign(plan=plan), "plan"))

for name, f in [("MSTL", fc_mstl), ("Theta", fc_theta)]:
    print(f"{name:6} roster at the forecast: {round(bt.roster_cost()(f))} dollars a week | at the 72% point: {roster_at(f)}")

> **Check.** You should see MSTL at 4044 and 3718 dollars, and Theta at 3883 and 5845 dollars.

With the range, MSTL gives the cheapest roster. Theta's range is so wide that its 72% point puts too many agents in every hour.

## 6. A known event as an extra input

Larkfield's Christmas delivery days, from 12 to 23 December, bring more tickets: customers ask if their parcel will arrive in time. Grace knows these dates a year ahead. MSTL can pass an extra input to its trend forecaster if that forecaster accepts one. ETS does not; AutoARIMA does.

> **Predict.** At the origin of Sunday 21 December 2025, 23:00, will the delivery-days input lower the WAPE of the week?

In [ ]:
from statsforecast.models import AutoARIMA

def delivery_days(ds):
    return ((ds.dt.month == 12) & (ds.dt.day >= 12) & (ds.dt.day <= 23)).astype(float)

o = pd.Timestamp("2025-12-21 23:00")
train = tickets[tickets["ds"] <= o].tail(2016).assign(delivery=lambda d: delivery_days(d["ds"]))
week = tickets[(tickets["ds"] > o) & (tickets["ds"] <= o + pd.Timedelta(hours=168))].assign(delivery=lambda d: delivery_days(d["ds"]))

def wape(y, f):
    return round(100 * (y - f).abs().sum() / y.sum(), 1)

plain = StatsForecast(models=[MSTL(season_length=[24, 168], trend_forecaster=AutoARIMA())], freq="h")
f_plain = plain.forecast(df=train[["unique_id", "ds", "y"]], h=168)["MSTL"].clip(lower=0).to_numpy()
event = StatsForecast(models=[MSTL(season_length=[24, 168], trend_forecaster=AutoARIMA())], freq="h")
f_event = event.forecast(df=train[["unique_id", "ds", "y", "delivery"]], h=168,
                         X_df=week[["unique_id", "ds", "delivery"]])["MSTL"].clip(lower=0).to_numpy()
y = week["y"].reset_index(drop=True)
print("WAPE without the input:", wape(y, f_plain), "| with the delivery days:", wape(y, f_event))
daily = pd.DataFrame({"real": y.to_numpy(), "without": f_plain, "with": f_event}, index=week["ds"]).resample("D").sum()
print(daily.round(0).astype(int).to_string())

> **Check.** You should see `WAPE without the input: 42.2 | with the delivery days: 38.7`, and seven days from 22 to 28 December.

With the input, the forecast is higher on 22 and 23 December and drops after them, as the real tickets do. At the other 12 origins of the backtest, the input is 0 in the history and in the week, so it changes nothing. An extra input helps only when the history has examples of it.

## Next

You forecast a series with two seasons and checked its range. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m04/several-seasons-and-long-series/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Forecasting as regression](https://learning.nextia-ai.com/courses/forecasting/m05/forecasting-as-regression/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m04/several-seasons-and-long-series/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.